# СБОРНЫЙ ПРОЕКТ 2

# Построение дашборда для телекоммуникационной компании

Заказчик исследования: большая телекоммуникационная компания, которая оказывает услуги на территории всего СНГ

Описание: у компании стояла задача - определить текущий уровень потребительской лояльности, или NPS, сред клиентов России. Для этого клиентам задавали классический вопрос: "Оцените по шкале от 1 до 10 вероятность того, что вы порекомендуете компанию друзьям и знакомым". Компания провела опрос, и им требуется подготовить дашборд с его итогами. Большую базу данных они разворачивать не стали и выгрузили данные в SQLite. Чтобы оценить результаты опросы, компания делит оценки клиентов на 3 группы: от 9 до 10 баллов - "сторонники"; от 7 до 8 баллов - "нейтралы"; от 0 до 6 баллов - "критики". Итоговое значение NPS расчитывает по формуле: % "сторонников" - % "критиков". Показатель NPS варьируется от -100% (когда все клиенты "критики") до 100% (когда все клиенты лояльны к сервису).  

Задача: подготовить дашборд с итогами опроса и ответить на вопросы:
- Как распределены участники опроса по возрасту и полу? Каких пользователей больше: новых или старых? Пользователи из каких городов активнее участвовали в опросе?
- Какие группы пользователей наиболее лояльны к сервису? Какие менее?
- Какой общий NPS среди всех опрошенных?
- Как можно описать клиентов, которые относятся к группе cторонников?

Описание данных для исследования и построение дашборда:
Таблица user (содержит основную информацию о клиентах);
Таблица location (справочник территорий, в которых телеком-компания оказывает услуги);
Таблица age_segment (данные о возрастных сегментах клиентов);
Таблица traffic_segment (данные о выделяемых сегментах по объёму потребляемого трафика);
Таблица lifetime_segment (данные о выделяемых сегментах по количеству месяцев «жизни» клиента — лайфтайму).

In [1]:
import os
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

In [2]:
path_to_db_local = 'telecomm_csi.db'
path_to_db_platform = '/datasets/telecomm_csi.db'
path_to_db = None

if os.path.exists(path_to_db_local):
    path_to_db = path_to_db_local
elif os.path.exists(path_to_db_platform):
    path_to_db = path_to_db_platform
else:
    raise Exception('Файл с базой данных SQLite не найден!')

if path_to_db:
    engine = create_engine(f'sqlite:///{path_to_db}', echo=False)

In [3]:
query = """
    SELECT user_id,
    lt_day,
    CASE
        WHEN lt_day <= 365 THEN TRUE
        WHEN lt_day > 365 THEN FALSE
        ELSE NULL
    END as is_new,
    CASE WHEN age < 0 THEN 0 ELSE age END as age,
    CASE
        WHEN gender_segment = 1.0 THEN "ж"
        WHEN gender_segment = 0.0 THEN "м"
        ELSE 'другой'
    END AS gender_segment,
    os_name,
    cpe_type_name,
    loc.country,
    LOWER(TRIM(loc.city)) as city,
    CASE
        WHEN substr(ag.title, 4) = 'n/a' THEN 'другой'
        ELSE substr(ag.title, 4)
    END as age_segment,
    substr(tr.title, 4) as traffic_segment,
    substr(lt.title, 4) as lifetime_segment,
    CASE WHEN nps_score < 0 THEN 0 ELSE nps_score END as nps_score,
    CASE
        WHEN nps_score <= 6 THEN "критики"
        WHEN nps_score >= 7 and nps_score <= 8 THEN "нейтралы"
        WHEN nps_score >= 9 and nps_score <= 10 THEN "cторонники"
        ELSE 'другой'
    END as nps_group
from user as u
LEFT JOIN location AS loc ON u.location_id = loc.location_id
LEFT JOIN age_segment AS ag ON u.age_gr_id = ag.age_gr_id
LEFT JOIN traffic_segment AS tr ON u.tr_gr_id = tr.tr_gr_id
LEFT JOIN lifetime_segment AS lt ON u.lt_gr_id = lt.lt_gr_id;
"""

In [4]:
df = pd.read_sql(query, engine)
df.head()

,user_id,lt_day,is_new,age,gender_segment,os_name,cpe_type_name,country,city,age_segment,traffic_segment,lifetime_segment,nps_score,nps_group
0,A001A2,2320,0,45.0,ж,ANDROID,SMARTPHONE,Россия,Уфа,45-54,1-5,36+,10,cторонники
1,A001WF,2344,0,53.0,м,ANDROID,SMARTPHONE,Россия,Киров,45-54,1-5,36+,10,cторонники
2,A003Q7,467,0,57.0,м,ANDROID,SMARTPHONE,Россия,Москва,55-64,20-25,13-24,10,cторонники
3,A004TB,4190,0,44.0,ж,IOS,SMARTPHONE,Россия,РостовнаДону,35-44,0.1-1,36+,10,cторонники
4,A004XT,1163,0,24.0,м,ANDROID,SMARTPHONE,Россия,Рязань,16-24,5-10,36+,10,cторонники


In [5]:
df.to_csv('telecomm_csi_tableau.csv', index=False)